In [9]:
############################################################
# 1. 환경 설정 및 라이브러리 로드
# ----------------------------------------------------------
# - requests: HTTP 요청
# - BeautifulSoup: HTML 파싱
# - pandas: 데이터프레임 생성
# - tqdm: 진행률 시각화
# - re: 정규표현식 기반 텍스트 추출
############################################################

import os
import time
import re 
from urllib.parse import urljoin, urlparse, parse_qs

import pandas as pd
import requests

from bs4 import BeautifulSoup as bs
from tqdm import tqdm

In [2]:
############################################################
# 2. HTTP 세션 설정 및 공통 유틸 함수
# ----------------------------------------------------------
# - Session 재사용으로 성능 향상
# - User-Agent 설정으로 차단 방지
# - 간단한 레이트리밋 적용
############################################################

BASE = "https://gmnara.com"     # 대상 사이트 기본 도메인 

session = requests.Session()    # requests.Session() 생성, 동일 세션을 통해 쿠키/헤더/연결 재사용 
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept-Language": "ko-KR,ko;q=0.9,en;q=0.8",
})

def get_soup(url, sleep_sec=0.3):
    """GET 요청 후 BeautifulSoup 반환 (간단 레이트리밋 포함)."""
    r = session.get(url, timeout=20)
    r.raise_for_status()    # HTTP 오류 예외처리
    time.sleep(sleep_sec)
    return bs(r.text, "html.parser")

In [3]:
############################################################
# 3. 목록 페이지에서 상세 링크 수집
# ----------------------------------------------------------
# - "업소정보" 버튼 탐색
# - javascript:openPopup(...) 패턴 처리
# - 상세페이지의 uid를 external_id로 사용
############################################################

# openPopup(...) 경로를 추출하는 정규표현식
POPUP_RE = re.compile(r"openPopup\('([^']+)'\)")

def extract_popup_path_from_href(href: str):
    """
    목록 페이지의 a태그 href 값에서 실제 상세페이지 경로를 추출.
    케이스: 
    1) javascript:openPopup('/?m=bbs&bid=cp&mod=view&uid=20259367&disp=popup');
    2) 직접 URL 케이스
    """
    if not href:
        return None

    # 케이스 1) 이면 괄호 안 실제 경로 반환
    m = POPUP_RE.search(href)
    if m:
        return m.group(1)

    # 케이스 2) 이면 그대로 반환 
    if "mod=view" in href and "uid=" in href:
        return href

    return None

def extract_uid_from_popup_url(popup_url: str):
    """
    상세페이지 URL에서 uid 값 추출
    사이트 내부에서 각 업소를 구분하는 고유값을 얻어 중복을 방지함
    """
    qs = parse_qs(urlparse(popup_url).query)    # 쿼리 문자열 추출해 dict로 변환
    uid = qs.get("uid", [None])[0]
    return uid

In [4]:
############################################################
# 4. 목록 페이지 파싱
# ----------------------------------------------------------
# - "업소정보" 버튼 탐색
# - 카드 영역에서 업소명/주소 후보 추출
# - uid 기준 중복 방지
############################################################

# 대한민국 광역단위 행정구역명으로 시작하는 패턴을 주소로 추정
ADDRESS_HINT_RE = re.compile(
    r"(서울|경기|인천|부산|대전|대구|광주|울산|세종|강원|충북|충남|전북|전남|경북|경남|제주)\s+[^\n]{5,80}"
)

def parse_listing_page(listing_url: str):
    """
    지역 목록 페이지 1개를 입력받아,
    해당 페이지에 포함된 업소들의 기본 정보를 리스트로 반환함.
    기본 정보: 리스트 url, 상세정보 url, uid, 업소명, 주소
    """
    soup = get_soup(listing_url)

    a_tags = soup.find_all("a", string=lambda s: s and "업소정보" in s)

    by_uid = {}     # key: external_id(uid), value: 업소정보 dict 

    # 각 업소정보 버튼 순회
    for a in a_tags:
        href = a.get("href", "")
        popup_path = extract_popup_path_from_href(href)
        if not popup_path:
            continue    # 상세페이지 아닌 경우 스킵 

        detail_url = urljoin(BASE, popup_path)  # 절대경로로 변환
        external_id = extract_uid_from_popup_url(detail_url)

        li = a.find_parent("li")    # 업소 카드 단위가 <li> 구조 안에 존재 

        # --- 업소명 후보 ---
        name_raw = None
        if li:
            title_el = li.select_one("div.title") or li.select_one(".title")
            if title_el:
                name_raw = title_el.get_text(" ", strip=True)   # 업소명의 공백 제거 

        # --- 주소 후보(card) ---
        address_raw_card = None
        if li:
            # 1차 시도: 구조 기반
            loc_el = li.select_one("div.location span") or li.select_one(".location span")
            if loc_el:
                address_raw_card = loc_el.get_text(" ", strip=True)
            else:
                # 2차 시도: 텍스트 기반 
                li_text = li.get_text("\n", strip=True)
                for line in li_text.split("\n"):
                    if ADDRESS_HINT_RE.search(line):
                        address_raw_card = line.strip()
                        break

        # --- uid 기준 중복 병합 ---
        if external_id not in by_uid:
            # 처음 등장한 uid라면 신규 생성 
            by_uid[external_id] = {
                "listing_url": listing_url,
                "detail_url": detail_url,
                "external_id": external_id,
                "name_raw": name_raw,
                "address_raw_card": address_raw_card,
            }
        else:
            # 있는 uid면 "더 좋은 값"을 merge (비어있을 때만 채우기)
            if (not by_uid[external_id].get("name_raw")) and name_raw:
                by_uid[external_id]["name_raw"] = name_raw
            if (not by_uid[external_id].get("address_raw_card")) and address_raw_card:
                by_uid[external_id]["address_raw_card"] = address_raw_card

            # detail_url이 더 정상적인 형태로 들어오면 업데이트(안전장치)
            if by_uid[external_id].get("detail_url") != detail_url:
                by_uid[external_id]["detail_url"] = detail_url

    return list(by_uid.values())


In [5]:
############################################################
# 5. 상세 페이지 정규표현식 파싱
# ----------------------------------------------------------
# - 텍스트 기반 필드 추출
# - 구조 변경 대응 가능 설계
############################################################

# 공백(일반 공백/특수공백) 섞여도 잡히게 \s* 허용
# 예: "업 종 테 마", "업종테마", "업종 테마"
LABEL_PATTERNS = {
    "category_raw": r"업\s*종\s*테마\s*:\s*(.+)",
    "name_detail": r"업\s*체\s*이름\s*:\s*(.+)",
    "phone": r"전\s*화\s*번호\s*:\s*(.+)",
    "hours": r"영\s*업\s*시간\s*:\s*(.+)",
    "tagline": r"한\s*줄\s*소\s*개\s*(?:\n|:)\s*(.+)",

    # 주소 후보 2개 case 
    "address_by_location": r"업\s*소\s*위\s*치\s*:\s*(.+)",
    "address_by_region": r"지\s*역\s*:\s*(.+)",
}

def parse_detail_popup(detail_url: str):
    """
    상세 팝업 페이지를 파싱하여
    텍스트 기반으로 주요 필드를 추출.
    """

    # 상세 페이지 요청
    soup = get_soup(detail_url)

    # 페이지 전체 텍스트 줄 단위 추출, 공백 정리 
    text = soup.get_text("\n", strip=True)
    text = re.sub(r"[ \t]+", " ", text)

    # 라벨 기반 정규표현식 추출 
    out = {}
    for k, pat in LABEL_PATTERNS.items():
        m = re.search(pat, text)
        out[k] = m.group(1).strip() if m else None
    # "업소위치" 또는 "지역" 통합
    out["address_raw"] = out.get("address_by_location") or out.get("address_by_region")

    # 지도 링크 탐색 -> href 근거 
    map_url_raw = None
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if any(x in href.lower() for x in ["map", "kakao", "naver", "google"]):
            map_url_raw = urljoin(BASE, href)
            break

    out["map_url_raw"] = map_url_raw
    return out


In [6]:
############################################################
# 6. Raw 스키마 정의 및 Row 생성
############################################################

# 컬럼 설명:
# - source_site: 데이터 출처 도메인
# - crawled_at: 수집 시각 (재현성 및 이력 관리 목적)
# - listing_url: 해당 업소가 발견된 목록 페이지 URL
# - detail_url: 실제 상세 페이지 URL
# - external_id: 사이트 내부 고유 식별자(uid)
# - name_raw: 원본 업소명 (정규화 전)
# - category_raw: 원본 업종 정보
# - address_raw: 원본 주소 텍스트
# - map_url_raw: 지도 링크 원본
# - status_raw: 상세 파싱 중 오류 발생 시 상태 기록
RAW_COLS = [
    "source_site", "crawled_at", "listing_url", "detail_url", "external_id",
    "name_raw", "category_raw", "address_raw", "map_url_raw", "status_raw"
]

# 데이터 수집 시점 기록 
def now_kst_str():
    return time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

def build_raw_row(listing_url, detail_url, external_id, name_raw_card=None, address_raw_card=None):
    '''
    하나의 업소에 대해 raw 레코드 생성
    '''
    detail = parse_detail_popup(detail_url)

    name_raw = detail.get("name_detail") or name_raw_card
    category_raw = detail.get("category_raw")
    address_raw = detail.get("address_raw") or address_raw_card

    row = {
        "source_site": "gmnara.com",
        "crawled_at": now_kst_str(),
        "listing_url": listing_url,
        "detail_url": detail_url,
        "external_id": external_id,
        "name_raw": name_raw,
        "category_raw": category_raw,
        "address_raw": address_raw,
        "map_url_raw": detail.get("map_url_raw"),
        "status_raw": None,
    }
    return row

In [11]:
############################################################
# 7-1. 실행 준비
# ############################################################

# 수집 대상 지역 목록 페이지 
listing_urls = [
    "https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8",  # 강남
    "https://gmnara.com/b/cp?area=%EA%B0%95%EB%B6%81",   # 강북
    "https://gmnara.com/b/cp?area=%EA%B4%80%EC%95%85",  # 관악
    "https://gmnara.com/b/cp?area=%EA%B5%AC%EB%A1%9C",  # 구로
    "https://gmnara.com/b/cp?area=%EB%85%B8%EC%9B%90",  # 노원
    "https://gmnara.com/b/cp?area=%EB%8F%99%EB%8C%80%EB%AC%B8",     # 동대문
    "https://gmnara.com/b/cp?area=%EB%A7%88%ED%8F%AC",      # 마포
    "https://gmnara.com/b/cp?area=%EC%84%B1%EB%B6%81",      # 성북
    "https://gmnara.com/b/cp?area=%EC%96%91%EC%B2%9C",      # 양천
    "https://gmnara.com/b/cp?area=%EC%9D%80%ED%8F%89",  # 은평
    "https://gmnara.com/b/cp?area=%EC%A4%91%EA%B5%AC",  # 중구
    "https://gmnara.com/b/cp?area=%EA%B0%95%EB%8F%99",  # 강동
    "https://gmnara.com/b/cp?area=%EA%B0%95%EC%84%9C",  # 강서
    "https://gmnara.com/b/cp?area=%EA%B4%91%EC%A7%84",  # 광진
    "https://gmnara.com/b/cp?area=%EA%B8%88%EC%B2%9C",  # 금천
    "https://gmnara.com/b/cp?area=%EB%8F%84%EB%B4%89",  # 도봉
    "https://gmnara.com/b/cp?area=%EB%8F%99%EC%9E%91",  # 동작
    "https://gmnara.com/b/cp?area=%EC%84%9C%EC%B4%88",  # 서초
    "https://gmnara.com/b/cp?area=%EC%86%A1%ED%8C%8C",  # 송파
    "https://gmnara.com/b/cp?area=%EC%98%81%EB%93%B1%ED%8F%AC",  # 영등포
    "https://gmnara.com/b/cp?area=%EC%A2%85%EB%A1%9C",   # 종로
    "https://gmnara.com/b/cp?area=%EC%A4%91%EB%9E%91"   #중랑 
]

# 목록 페이지에서 업소 기본 정보 수집(상세 url, uid, 업소명/주소 후보)
all_list_items = []
for u in listing_urls:
    all_list_items.extend(parse_listing_page(u))

print("list items:", len(all_list_items))   # 수집된 업소 수 확인 
pd.DataFrame(all_list_items).head(3)

list items: 34


,listing_url,detail_url,external_id,name_raw,address_raw_card
0,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,10,[홈타이] 영앤핫 홈케어,"서울 강남구 강남대로 지하 396 (역삼동) 서울,경기,인천"
1,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259629,[로드샵] 히트스파,서울 강남구 봉은사로 129-1 신논현 3번출구
2,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20252610,[로드샵] 강남일급수스웨디시,서울 강남구 학동로 지하 102 (논현동) 논혁역7번출구


In [12]:
############################################################
# 7-2. 실행
############################################################

rows = []
for item in tqdm(all_list_items):
    try:
        # 정상 흐름: 상세 페이지 파싱 -> raw 스키마에 맞는 dict 생성 
        rows.append(
            build_raw_row(
                listing_url=item["listing_url"],
                detail_url=item["detail_url"],
                external_id=item["external_id"],
                name_raw_card=item.get("name_raw"),
                address_raw_card=item.get("address_raw_card"),
            )
        )
    except Exception as e:
        # 예외처리: 실패 로그를 남기고 계속 진행
        rows.append({
            "source_site": "gmnara.com",
            "crawled_at": now_kst_str(),
            "listing_url": item["listing_url"],
            "detail_url": item["detail_url"],
            "external_id": item["external_id"],
            "name_raw": item.get("name_raw"),
            "category_raw": None,
            "address_raw": item.get("address_raw_card"),
            "map_url_raw": None,
            "status_raw": f"ERROR: {type(e).__name__}",     # 오류 유형 
        })

df_raw = pd.DataFrame(rows, columns=RAW_COLS)   # rows 리스트 -> dataFrame 
df_raw.head(5)

100%|██████████| 34/34 [00:12<00:00,  2.66it/s]


,source_site,crawled_at,listing_url,detail_url,external_id,name_raw,category_raw,address_raw,map_url_raw,status_raw
0,gmnara.com,2026-02-26 10:12:52,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,10,영앤핫 홈케어,홈타이,"서울 강남구 강남대로 지하 396 (역삼동) 서울,경기,인천",https://www.google.co.kr/search?q=건마나라,None
1,gmnara.com,2026-02-26 10:12:52,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259629,히트스파,로드샵,서울 강남구 봉은사로 129-1 신논현 3번출구,https://www.google.co.kr/search?q=건마나라,None
2,gmnara.com,2026-02-26 10:12:53,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20252610,강남일급수스웨디시,로드샵,서울 강남구 학동로 지하 102 (논현동) 논혁역7번출구,https://www.google.co.kr/search?q=건마나라,None
3,gmnara.com,2026-02-26 10:12:53,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259584,5월스파,로드샵,서울 강남구 학동로33길 15 (논현동),https://www.google.co.kr/search?q=건마나라,None
4,gmnara.com,2026-02-26 10:12:53,https://gmnara.com/b/cp?area=%EA%B0%95%EB%82%A8,https://gmnara.com/?m=bbs&bid=cp&mod=view&uid=...,20259472,강남 vip스웨디시&왁싱샵,로드샵,서울 강남구 일원로 지하 2 (일원동) 대청역 4번출구로 나와서 직진,https://www.google.co.kr/search?q=건마나라,None


In [13]:
############################################################
# 8. csv 형태로 저장
############################################################

from datetime import datetime

# 저장 경로
out_dir = "../data"
os.makedirs(out_dir, exist_ok=True)

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(out_dir, f"gmnara_raw_{timestamp}.csv")

df_raw.to_csv(out_path, index=False, encoding="utf-8-sig")

print("saved:", os.path.abspath(out_path))

saved: c:\workspace\DSJA_TeamProject\sbsDSJA_4th_team6_5W1H\crwaling\data\gmnara_raw_20260226_101312.csv
